In [1]:
from prometheus_pandas import query
import datetime as dt
from zoneinfo import ZoneInfo
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

local_address = "nova-11.lyon.grid5000.fr"
local_port = "30090"
p = query.Prometheus(f"http://admin:prom-operator@{local_address}:{local_port}")    

local_tz = ZoneInfo("Europe/Paris")

def execute_query_range(experiment, metric_query, step, output_filename, plot=False):
    tput_series = []

    print("\nBeginning request execution")
    for label, config in experiment.items():    
        df_metric = p.query_range(
            query=metric_query,
            start=config['start'].isoformat(),
            end=config['end'].isoformat(),
            step=step
        )

        tput_series.append(df_metric)

    if tput_series:
        combined_df = pd.concat(tput_series, axis=1)
        combined_df.to_csv(output_filename, index_label='Time')
        print(f"\nFile saved : '{output_filename}'")
        if plot:
            plt.figure(figsize=(12, 8))
            ax = combined_df.boxplot(grid=True, patch_artist=True)
            plt.title(metric_query)
            plt.show()
    else:
        print("\nRequest Error")

In [11]:
exp = {
    'rocks': {
        'start': dt.datetime(2025, 7, 31, 13, 57, 30, tzinfo=local_tz),
        'end': dt.datetime(2025, 7, 31, 13, 58, 30, tzinfo=local_tz)
    }
}


In [12]:
metric_query = 'flink_jobmanager_job_uptime'
step = "1s"
output_filename = 'exp.csv'
execute_query_range(exp, metric_query, step, output_filename)



Beginning request execution

File saved : 'exp.csv'


### BoxPlot

In [ ]:
file_path = 'exp2b_q11_sst_size.csv'

try:
    df = pd.read_csv(file_path, index_col=0)
    
    df.columns = ['rocks', 'forst', 'forst_small_cache', 'forst_no_cache']
    #df = df.iloc[:, :-1] 
    plt.figure(figsize=(10, 7))
    df.boxplot(grid=True, patch_artist=True)
    
    plt.title('SST Files size for q11 (10min execution)')
    plt.ylabel('SST size (B)')
    plt.xlabel('Configuration')
    #plt.yticks(np.arange(0.6, 1, 0.1))
    plt.savefig('boxplot_q11_sst_size.png')
    plt.show()
    
except FileNotFoundError:
    print(f"Le fichier '{file_path}' est introuvable.")
except Exception as e:
    print(f"Une erreur est survenue : {e}")

Le fichier 'exp2b_q11_sst_size.csv' est introuvable.
